In [1]:
!pip install langchain "langchain[openai]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 5.7 MB/s eta 0:00:00


In [9]:
import os
from langchain_openai import AzureChatOpenAI
from google.colab import userdata

os.environ["AZURE_OPENAI_API_KEY"] = userdata.get('AZURE_OPENAI_API_KEY')
os.environ["AZURE_OPENAI_ENDPOINT"] =userdata.get("AZURE_OPENAI_ENDPOINT")
os.environ["OPENAI_API_VERSION"] = "2024-12-01-preview"

model = AzureChatOpenAI(
    model="gpt-5.4-mini",
    azure_deployment="gpt-5.4-mini"
)

In [3]:
from langchain_core.tools import tool
@tool
def estimate_latency(components: list[int]) -> dict:
    """
    Estimate total end-to-end latency for a system design by summing
    individual component latencies.

    Use this tool when the user provides latency values for multiple
    components (e.g., auth, database, cache, API gateway) and wants
    to know the total latency in milliseconds.

    Args:
        components: A list of integer latency values in milliseconds.
                    Example: [40, 80, 25, 30] for
                    auth=40ms, DB=80ms, cache=25ms, queue=30ms.

    Returns:
        A dict with total latency, component count, and breakdown.
    """
    total = sum(components)
    return {
        "total_latency_ms": total,
        "component_count": len(components),
        "breakdown": components
    }

In [4]:
prompt = """
    You are a senior backend architecture reviewer.

Your role:
- Review technical designs using the conversation context.
- Use the available tool(estimate_latency) only when a calculation or deterministic check is required.
- Prefer direct reasoning for conceptual questions and the tool for numeric estimation.

Rules:
1. Work only from the user’s input and conversation context.
2. Do not invent production facts, metrics, or infrastructure details.
3. If a question requires arithmetic or estimation, use the tool instead of mental math.
4. If assumptions are missing, list them clearly before answering.
5. Separate assumptions, analysis, and recommendation.
6. Keep answers concise, technical, and implementation-aware.
7. When the tool is used, incorporate the result into an engineering recommendation, not just a number.
8. Do not overuse the tool; call it only when it materially improves accuracy.
9. If the tool is insufficient, explain the limitation explicitly.
10. Output in this format:
   - Problem
   - Assumptions
   - Analysis
   - Recommendation"""

In [5]:
from langchain.agents import create_agent

agent = create_agent(model, system_prompt=prompt)

In [6]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Design a checkout service that handles 5k requests/sec with retries and idempotency."
    }]
})

In [7]:
print(result)

{'messages': [HumanMessage(content='Design a checkout service that handles 5k requests/sec with retries and idempotency.', additional_kwargs={}, response_metadata={}, id='2d8457b8-c5d7-4272-a86b-ebcc333146de'), AIMessage(content='- **Problem**\n  Design a checkout service that can sustain **5k requests/sec**, while supporting **retries** and **idempotency** without double-charging or duplicate orders.\n\n- **Assumptions**\n  1. “Checkout” means creating an order and initiating payment in one logical API flow.\n  2. Clients may retry due to timeouts, network errors, or 5xx responses.\n  3. Requests may arrive concurrently with the same idempotency key.\n  4. We need at-least-once delivery from clients, but exactly-once *effect* for checkout.\n  5. A relational database or strongly consistent store is available for order and idempotency state.\n  6. Payment provider APIs are external and may also be retried safely only if given an idempotency key.\n  7. 5k RPS is sustained and requires h

In [8]:
print(result['messages'][-1].content)

- **Problem**
  Design a checkout service that can sustain **5k requests/sec**, while supporting **retries** and **idempotency** without double-charging or duplicate orders.

- **Assumptions**
  1. “Checkout” means creating an order and initiating payment in one logical API flow.
  2. Clients may retry due to timeouts, network errors, or 5xx responses.
  3. Requests may arrive concurrently with the same idempotency key.
  4. We need at-least-once delivery from clients, but exactly-once *effect* for checkout.
  5. A relational database or strongly consistent store is available for order and idempotency state.
  6. Payment provider APIs are external and may also be retried safely only if given an idempotency key.
  7. 5k RPS is sustained and requires horizontal scaling.

- **Analysis**
  1. **Core API shape**
     - `POST /checkout`
     - Client sends:
       - `Idempotency-Key` header
       - cart/order payload
       - user/session identifier
     - Server returns:
       - `201 Crea